<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m6_01_bagging_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M6.01 — Many trees are better than one

In the lab we saw that averaging many trees, each trained on a random
variation of the data, gives a better model than any single tree:
**bagging** and **random forests**. In this workbook you build a forest
**by hand** to predict the wages of football players, look at what a
bootstrap sample really contains, and discover a validation score that
a forest gives you for free.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "best_constant": "7896a617a785dd96",
    "better_model": "184df936d41b2da8",
    "default_max_features": "6b86b273ff34fce1",
    "forests_for_oob": "6b86b273ff34fce1",
    "share_absent": "00750752a946354e"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

The data comes from the video game *FIFA 22*. Each row is one of 8000
professional players as the game describes him: age, height, weight, an
`overall` rating and a `potential` rating out of 100, and about twenty
skill ratings (crossing, finishing, sprint speed, goalkeeping…). The
target, `wage_eur`, is the player's weekly wage in euros according to
the game.

For now we use the 27 numerical columns and leave `nationality_name`
aside.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

players = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/football_players.csv"
)
target = players["wage_eur"]
data = players.drop(columns=["wage_eur", "nationality_name"])
data_train, data_test, target_train, target_test = train_test_split(data, target, random_state=0)
print(f"{len(players)} players, {data.shape[1]} numerical features")
players[["age", "overall", "potential", "attacking_finishing", "movement_sprint_speed", "wage_eur"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(target, bins=80)
_ = ax.set(xlabel="weekly wage (euros)", ylabel="number of players", title="A few players earn far more than the rest")

### ➕ Going further: which baseline for a lopsided target?

Half of the players earn €3000 a week or less; a handful earn more than
€200 000. The **mean** wage is pulled up by the stars, the **median**
(the wage of the player in the middle) is not.

🔮 **Predict first.** We measure errors with the mean absolute error. A
baseline predicts the same wage for everybody. Which constant gives the
lower error: the `"mean"` or the `"median"` of the training wages?

In [ ]:
my_prediction = ____
commit(my_prediction, ["mean", "median"])

In [ ]:
baseline = {
    "mean": mean_absolute_error(target_test, np.full(len(target_test), target_train.mean())),
    "median": mean_absolute_error(target_test, np.full(len(target_test), target_train.median())),
}
print(f"mean wage   {target_train.mean():7.0f} €  ->  error {baseline['mean']:.0f} €")
print(f"median wage {target_train.median():7.0f} €  ->  error {baseline['median']:.0f} €")

✏️ Which constant is the right baseline when the error is the mean
absolute error?

In [ ]:
best_constant = ____
check("best_constant", best_constant)

The mean is the best constant for the **squared** error, the median for
the **absolute** error. With a lopsided target they are far apart, and
comparing a model with the wrong one flatters the model.
`DummyRegressor(strategy="median")` gives this baseline.

## 2. One tree

A single unlimited tree, as in module 5:

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=0).fit(data_train, target_train)
tree_error = mean_absolute_error(target_test, tree.predict(data_test))
print(f"single tree: error {tree_error:.0f} € per week")

Better than the baseline, and we know its weakness from workbooks M2.03
and M5.02: train it on slightly different players and it changes a lot.

## 3. A bootstrap sample

We cannot recruit 6000 new players to train a second tree. The trick of
bagging is to build a new training set **from the one we have**, by
drawing 6000 players at random **with replacement**: after each draw,
the player is put back and can be drawn again. This is a **bootstrap
sample**.

🔮 **Predict first.** In a bootstrap sample of 6000 draws out of 6000
players, some players appear several times and others never. Which
share of the players appears **at least once**? **"almost all"**,
**"about two thirds"** or **"about half"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["almost all", "about two thirds", "about half"])

In [ ]:
rng = np.random.default_rng(0)
n_train = len(data_train)
drawn = rng.integers(0, n_train, size=n_train)
share_present = len(np.unique(drawn)) / n_train
print(f"{n_train} draws, {len(np.unique(drawn))} different players: {share_present:.1%} of the training set")
print(f"most drawn player: {np.bincount(drawn).max()} times")

✏️ Which share of the players is **absent** from a bootstrap sample?
Write `"almost none"`, `"about one third"` or `"about half"`.

In [ ]:
share_absent = ____
check("share_absent", share_absent)

This holds for any large dataset: about 63% of the rows are present and
37% are left out. Each bootstrap sample is a different two thirds of the
players, with different repetitions: enough to make each tree different.

## 4. A forest by hand

We train 50 trees, each on its own bootstrap sample, and keep the
predictions of each one for the test players.

In [ ]:
tree_predictions = []
for _ in range(50):
    drawn = rng.integers(0, n_train, size=n_train)
    one_tree = DecisionTreeRegressor(random_state=0).fit(data_train.iloc[drawn], target_train.iloc[drawn])
    tree_predictions.append(one_tree.predict(data_test))
tree_predictions = np.array(tree_predictions)

single_errors = np.abs(tree_predictions - target_test.to_numpy()).mean(axis=1)
print(f"errors of the 50 trees taken one by one: from {single_errors.min():.0f} to {single_errors.max():.0f} €")

Each tree alone is about as good as the tree of section 2. Now we
**average** their predictions.

🔮 **Predict first.** The average of the 50 trees will have an error
**"about equal to the average tree"**, **"equal to the best of the 50
trees"** or **"lower than the best of the 50 trees"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["about equal to the average tree", "equal to the best of the 50 trees", "lower than the best of the 50 trees"])

In [ ]:
errors_of_average = [
    mean_absolute_error(target_test, tree_predictions[:n_trees].mean(axis=0)) for n_trees in range(1, 51)
]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, 51), single_errors, ".", color="gray", label="each tree alone")
ax.plot(range(1, 51), errors_of_average, color="black", linewidth=2, label="average of the first n trees")
ax.set(xlabel="number of trees", ylabel="error (€ per week)")
ax.legend()
print(f"best single tree:        {single_errors.min():.0f} €")
print(f"average of the 50 trees: {errors_of_average[-1]:.0f} €")

✏️ Which is better: the best of the 50 trees, or the average of all 50?
Write `"best single tree"` or `"average"`.

In [ ]:
better_model = ____
check("better_model", better_model)

No tree is as good as the committee. Each tree makes its own errors on
its own players; when the predictions are averaged, errors that go in
different directions cancel. This is the variance reduction of workbook
M2.03, and the curve shows the other half of the story: after 20 or 30
trees, adding more hardly helps.

🧠 **Think.** Would the same trick work with 50 **linear regressions**
trained on bootstrap samples? (Hint: do they make different errors?)

## 5. The same thing in scikit-learn

`BaggingRegressor` does exactly what we did. `RandomForestRegressor`
adds a second source of randomness: at each question, a tree may only
choose among a **random subset of the features** (`max_features`), which
makes the trees more different from one another.

In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor

forests = {
    "BaggingRegressor": BaggingRegressor(DecisionTreeRegressor(), n_estimators=50, random_state=0, n_jobs=2),
    "RandomForestRegressor (default)": RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=2),
    "RandomForestRegressor (max_features=0.3)": RandomForestRegressor(
        n_estimators=50, max_features=0.3, random_state=0, n_jobs=2
    ),
}
forest_errors = {}
for name, forest in forests.items():
    forest.fit(data_train, target_train)
    forest_errors[name] = mean_absolute_error(target_test, forest.predict(data_test))
pd.Series(forest_errors, name="error (€ per week)").round(0).to_frame()

✏️ By default, which share of the features may each question of a
`RandomForestRegressor` choose from? Enter `0.3` or `1`. (Look at the
table above: which of the two forests behaves like the bagging?)

In [ ]:
default_max_features = ____
check("default_max_features", default_max_features, hint="By default a regression forest lets every question look at all the features.")
print("default max_features of RandomForestRegressor:", RandomForestRegressor().get_params()["max_features"])

A surprise hidden in the defaults: for **regression**, scikit-learn's
random forest looks at all the features at every question, so it is a
plain bagging of trees. (For classification the default is the square
root of the number of features.) Whether a smaller `max_features` helps
depends on the data, and workbook M6.03 treats it as a hyperparameter to
tune.

## 6. ➕ Going further: a validation score for free

Each tree of a forest is trained on about two thirds of the players. The
remaining third, the players **out of its bag**, has never been seen by
that tree. So each player can be predicted by the trees that did not
train on him, and the forest can be evaluated **without setting any
data aside**. With `oob_score=True`, scikit-learn does this during
training.

🔮 **Predict first.** We compare this out-of-bag score with a 5-fold
cross-validation of the same forest (both are R² scores: 1 is perfect,
0 is the level of predicting the mean). Will they be **"very
different"** or **"about the same"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["very different", "about the same"])

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

oob_forest = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=0, n_jobs=2).fit(data, target)
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=2),
    data,
    target,
    cv=KFold(n_splits=5, shuffle=True, random_state=0),
)
print(f"out-of-bag score (1 forest trained):         {oob_forest.oob_score_:.3f}")
print(f"5-fold cross-validation (5 forests trained): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

✏️ How many forests had to be trained to obtain the out-of-bag score?

In [ ]:
forests_for_oob = ____
check("forests_for_oob", forests_for_oob)

The out-of-bag score costs nothing more than the training itself and
agrees with cross-validation. It is a property of **bootstrap** methods
only: it exists because each tree leaves players out.

🧠 **Think.**

1. A player is in the bag of about 63 trees out of 100. How many trees
   predict him for the out-of-bag score? Is this score based on a forest
   as large as the full one?
2. Why can a gradient boosting model (next workbook), where every tree
   is trained on all the players, not have an out-of-bag score?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - why the median, and not the mean, is the baseline for this target;
   - what a bootstrap sample is and which share of the players it
     contains;
   - the forest by hand: why the average beats the best single tree;
   - the out-of-bag score: where it comes from and what it saves.